# Error Taxonomy Demo

This notebook explores linguistic error categories produced by the TranslationErrorAnalyzer and summarizes error distributions.

In [1]:
from pathlib import Path
import os
import sys

import pandas as pd

def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'config.py').exists() and (candidate / 'analysis').exists():
            return candidate
    raise RuntimeError('Could not locate Machine_Translation project root.')

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from analysis.error_analyzer import TranslationErrorAnalyzer

print(f'Project root: {PROJECT_ROOT}')

Project root: C:\Users\Nikolai\OneDrive\Desktop\Portfolio\CS_Language_Portfolio\projects\Machine_Translation


In [2]:
examples = [
    {
        'id': 'negation_loss_1',
        'source': 'The software is not working correctly.',
        'target': 'Die Software funktioniert richtig.',
        'reference': 'Die Software funktioniert nicht korrekt.',
        'source_lang': 'en',
        'target_lang': 'de'
    },
    {
        'id': 'word_order_1',
        'source': 'I have never seen this before.',
        'target': 'Ich habe dieses gesehen zuvor nie.',
        'reference': 'Ich habe so etwas noch nie gesehen.',
        'source_lang': 'en',
        'target_lang': 'de'
    },
    {
        'id': 'agreement_1',
        'source': 'Der grosse Mann sieht den kleinen Jungen.',
        'target': 'Der grosser Mann sieht den klein Jungen.',
        'reference': 'Der grosse Mann sieht den kleinen Jungen.',
        'source_lang': 'de',
        'target_lang': 'de'
    },
    {
        'id': 'compound_1',
        'source': 'Der Schmetterlingshaus ist sehr gross.',
        'target': 'The butterfly home is very big.',
        'reference': 'The butterfly house is very large.',
        'source_lang': 'de',
        'target_lang': 'en'
    }
]

pd.DataFrame(examples)[['id', 'source_lang', 'target_lang', 'source', 'target']]

,id,source_lang,target_lang,source,target
0,negation_loss_1,en,de,The software is not working correctly.,Die Software funktioniert richtig.
1,word_order_1,en,de,I have never seen this before.,Ich habe dieses gesehen zuvor nie.
2,agreement_1,de,de,Der grosse Mann sieht den kleinen Jungen.,Der grosser Mann sieht den klein Jungen.
3,compound_1,de,en,Der Schmetterlingshaus ist sehr gross.,The butterfly home is very big.


In [3]:
analyzer = TranslationErrorAnalyzer()
analysis_objects = []
for row in examples:
    analysis = analyzer.analyze(
        source=row['source'],
        target=row['target'],
        reference=row['reference'],
        source_lang=row['source_lang'],
        target_lang=row['target_lang']
    )
    analysis_objects.append((row['id'], analysis))

len(analysis_objects)

4

In [4]:
error_rows = []
for example_id, analysis in analysis_objects:
    for err in analysis.errors:
        error_rows.append({
            'example_id': example_id,
            'category': err.category.value,
            'severity': err.severity,
            'confidence': round(err.confidence, 2),
            'explanation': err.explanation
        })

errors_df = pd.DataFrame(error_rows)
errors_df.sort_values(['category', 'severity', 'confidence'], ascending=[True, True, False])

,example_id,category,severity,confidence,explanation
17,compound_1,compound_breakdown,low,0.50,Compound 'Schmetterlingshaus' may have decompo...
1,negation_loss_1,lexical_mismatch,high,0.75,Negation lost in translation
5,word_order_1,lexical_mismatch,high,0.75,Negation lost in translation
14,compound_1,morphological_loss,medium,0.60,Morphological feature lost: gender:m
0,negation_loss_1,oov_untranslated,high,0.80,Word 'software' appears untranslated or OOV in...
9,agreement_1,oov_untranslated,high,0.80,Word 'Der' appears untranslated or OOV in target
10,agreement_1,oov_untranslated,high,0.80,Word 'Mann' appears untranslated or OOV in target
11,agreement_1,oov_untranslated,high,0.80,Word 'sieht' appears untranslated or OOV in ta...
12,agreement_1,oov_untranslated,high,0.80,Word 'den' appears untranslated or OOV in target
13,agreement_1,oov_untranslated,high,0.80,Word 'Jungen.' appears untranslated or OOV in ...


In [5]:
category_counts = errors_df.groupby('category').size().reset_index(name='count').sort_values('count', ascending=False)
severity_counts = errors_df.groupby('severity').size().reset_index(name='count').sort_values('count', ascending=False)

category_counts, severity_counts

(             category  count
 3    oov_untranslated      6
 4      semantic_drift      5
 5    word_order_shift      3
 1    lexical_mismatch      2
 0  compound_breakdown      1
 2  morphological_loss      1,
   severity  count
 0     high     10
 1      low      4
 2   medium      4)

In [6]:
# Inspect one full structured output for downstream serialization or API usage.
analysis_objects[0][1].to_dict()

{'source': 'The software is not working correctly.',
 'target': 'Die Software funktioniert richtig.',
 'reference': 'Die Software funktioniert nicht korrekt.',
 'source_lang': 'en',
 'target_lang': 'de',
 'errors': [{'category': 'oov_untranslated',
   'source_span': 'software',
   'target_span': 'software',
   'reference_span': None,
   'severity': 'high',
   'explanation': "Word 'software' appears untranslated or OOV in target",
   'confidence': 0.8},
  {'category': 'lexical_mismatch',
   'source_span': 'The software is not working correctly.',
   'target_span': 'Die Software funktioniert richtig.',
   'reference_span': None,
   'severity': 'high',
   'explanation': 'Negation lost in translation',
   'confidence': 0.75},
  {'category': 'word_order_shift',
   'source_span': 'The software is not working correctly.',
   'target_span': 'Die Software funktioniert richtig.',
   'reference_span': None,
   'severity': 'low',
   'explanation': 'Word order significantly reordered (similarity: 0

## How To Use This

- Extend the examples list with real model outputs.
- Track category trends over model versions.
- Combine this taxonomy with BLEU/chrF in the evaluation notebook.